```
# Lab type: prompt
# Course: ML301 — Deep Learning with PyTorch
# Lesson: Transfer Learning
# Task: This lab has three parts. Parts 1 and 2 ask you to write prompts for an AI tool, paste the output, and audit it. Part 3 gives you a pre-written AI response to audit directly.
```

In [ ]:
import torch
import torch.nn as nn
# Note: torchvision required for pretrained models
# !pip install torchvision  # Uncomment if needed

print("Transfer learning lab — no code to run until you paste AI output below.")
print("Read each task description carefully before writing your prompt.")

## Dataset Context

You are adapting a pretrained ResNet-18 (trained on ImageNet) for a **medical image binary classification task**: classifying chest X-rays as pneumonia positive or negative.

- Training set: 1,200 images (small)
- Validation set: 300 images
- Images: 224×224 RGB (grayscale converted to 3-channel)
- Label: 0 = normal, 1 = pneumonia

This domain is different from ImageNet (natural photos). Dataset is small. Use this context in every prompt.

## Task 1: Freeze / Unfreeze Strategy

### Step 1: Weak prompt (do not use this)

The following is an example of what **not** to write:

> *'Write PyTorch code to fine-tune a ResNet-18 for image classification.'*

**Why it's weak:** No freeze/unfreeze specification, no mention of dataset size, no domain info, no differential learning rate requirement. AI will likely unfreeze everything with a uniform learning rate.

### Step 2: Write your own strong prompt

Your prompt should specify:
1. The pretrained model (ResNet-18, ImageNet weights)
2. The freeze strategy appropriate for a small, different-domain dataset (freeze backbone, train head only)
3. That the new head should be `nn.Linear(512, 1)` (binary output)
4. The learning rate (1e-3 for the head)
5. That the model must be set to evaluation mode before any inference

Write your strong prompt here:

---

*(Your prompt here)*

---

<details>
<summary>🔑 Model prompt — Task 1</summary>

**Example strong prompt:**

> Write PyTorch code to fine-tune a ResNet-18 model (pretrained on ImageNet) for binary chest X-ray classification (pneumonia vs. normal). Dataset: 1,200 training images, 300 validation images — small and out-of-domain relative to ImageNet.
>
> Requirements:
> 1. Freeze all backbone parameters so only the classification head is trained.
> 2. Replace the existing `fc` layer with `nn.Linear(512, 1)` (binary output, no sigmoid — I'll use `BCEWithLogitsLoss`).
> 3. Set learning rate to 1e-3 for the head only.
> 4. The model must be set to evaluation mode (`model.eval()`) before any inference or validation step.
>
> Show the full model setup, optimizer definition, and a single training epoch loop.

**Why it's strong:** Specifies the pretrained model and weights, dataset size and domain, the exact freeze strategy, the head architecture, the loss function, the learning rate, and the evaluation-mode requirement. The AI has no room to guess.

</details>

### Step 3: Paste AI output here

Run your prompt in an AI tool (Claude, ChatGPT, Copilot, etc.) and paste the generated code below.

In [ ]:
# Paste AI-generated code here (Task 1: freeze strategy)
# Then run this cell to verify it works

### Step 4: Audit the AI output

Check each item and mark ✓ or ✗:

1. [ ] All backbone parameters have `requires_grad = False` before the head is replaced
2. [ ] The replacement head is `nn.Linear(512, 1)` (ResNet-18 fc has 512 in_features)
3. [ ] The new head has `requires_grad = True` (default for new layers — confirm it was not frozen)
4. [ ] The training loop uses `BCEWithLogitsLoss` (not `BCELoss` on sigmoid output)
5. [ ] The model is set to evaluation mode before any inference or validation step

**If any item fails:** Note the failure and correct it manually in the cell above.

## Task 2: Learning Rate for Backbone vs Head

### Step 1: Weak prompt (do not use this)

> *'What is the best learning rate for fine-tuning ResNet-18?'*

**Why it's weak:** No mention of differential LR, no context about which layers are frozen, no dataset size. AI returns a single learning rate.

### Step 2: Write your strong prompt

Extend Task 1: unfreeze the last two conv blocks (layer3 and layer4 in ResNet-18). Your prompt should ask the AI to:
1. Set up two parameter groups: backbone (layer3+layer4) with lr=1e-5, head with lr=1e-3
2. Use `torch.optim.Adam` with these two parameter groups
3. Explain why the backbone needs a lower learning rate than the head

Write your strong prompt here:

---

*(Your prompt here)*

---

<details>
<summary>🔑 Model prompt — Task 2</summary>

**Example strong prompt:**

> Extend the ResNet-18 fine-tuning setup from Task 1. Unfreeze the last two convolutional blocks (`layer3` and `layer4`) for partial fine-tuning while keeping earlier layers frozen.
>
> Requirements:
> 1. Create two optimizer parameter groups using `torch.optim.Adam`:
>    - Backbone group (`layer3` + `layer4` parameters): `lr=1e-5`
>    - Head group (`fc` parameters): `lr=1e-3`
> 2. In your explanation, state why the backbone needs a learning rate at least 10× lower than the head (catastrophic forgetting risk).
> 3. Do not apply a single uniform learning rate to all parameters.
>
> Show the parameter group setup and optimizer instantiation.

**Why it's strong:** Specifies exactly which layers to unfreeze, the exact lr values for each group, the required ratio, the optimizer, and asks for a justification of the differential lr. Eliminates the AI's most common failure mode (single uniform lr).

</details>

### Step 3: Paste AI output here

In [ ]:
# Paste AI-generated code here (Task 2: differential learning rates)

### Step 4: Audit the AI output

Check each item:

1. [ ] Two optimizer parameter groups are created (not a single uniform lr)
2. [ ] Backbone parameter group lr ≤ 1e-5 (at least 10× lower than head lr)
3. [ ] Head parameter group lr = 1e-3 (or close — the ratio matters, not the exact value)
4. [ ] The explanation correctly identifies catastrophic forgetting as the risk of a high backbone lr
5. [ ] The code unfreezes layer3 and layer4 specifically (not all backbone layers)

**Note any failures and correct them.**

## Task 3: Audit a Pre-Written AI Response

The following code was generated by an AI tool given this prompt:

> *'Write PyTorch code to fine-tune ResNet-18 on a small medical image dataset for binary classification.'*

**Your job:** Read the code carefully and identify the errors. There are exactly **3 errors**. Do not run the code — audit by inspection.

In [ ]:
# AI-generated code — do NOT modify this cell. Audit it in the markdown cell below.

import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim

# Load pretrained ResNet-18
model = models.resnet18(weights='IMAGENET1K_V1')

# Freeze all backbone layers
for param in model.parameters():
    param.requires_grad = False

# Replace classification head
model.fc = nn.Linear(512, 1)

# ERROR: single learning rate for all parameters
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch(model, loader, criterion, optimizer):
    # ERROR: model is never switched to training mode here
    # (model.train() is missing — assume set-to-eval was called earlier)
    total_loss = 0
    for inputs, labels in loader:
        optimizer.zero_grad()
        outputs = model(inputs).squeeze(1)
        loss = criterion(outputs, labels.float())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

criterion = nn.BCEWithLogitsLoss()
# (caller would invoke train_one_epoch here)

### Your Audit

For each error: describe what it is, why it is wrong, and what the correct code should be.

---

**Error 1:**

*(Write here)*

---

**Error 2:**

*(Write here)*

---

**Error 3:**

*(Write here)*

---

<details>
<summary>🔑 Reveal errors — Task 3</summary>

**Error 1 — Single uniform learning rate:**
`optim.Adam(model.parameters(), lr=1e-3)` passes all parameters at the same learning rate. For pure feature extraction this is benign (frozen backbone params produce zero gradients). But the code provides no structure for differential learning rates — if the backbone is later unfrozen, all layers will train at the same high rate, risking catastrophic forgetting. Correct: `optim.Adam(model.fc.parameters(), lr=1e-3)` for feature extraction, or two parameter groups for partial fine-tuning.

**Error 2 — No differential learning rate structure:**
Directly related to Error 1: there is no mechanism to assign different rates to backbone and head. Any extension to partial fine-tuning will fail to protect pretrained features. Should be: `optim.Adam([{"params": model.fc.parameters(), "lr": 1e-3}])` at minimum, or two groups with separate backbone lr.

**Error 3 — `model.train()` never called before the training loop:**
The comment acknowledges that `model.eval()` was called earlier, but `train_one_epoch` never calls `model.train()`. BatchNorm therefore uses frozen running statistics (wrong for training) and Dropout is inactive (no regularisation). `model.train()` must be called at the start of each epoch, before the batch loop.

</details>

<details>
<summary>🔑 Instructor notes — Task 3 errors</summary>

**Error 1:** The optimizer is passed `model.parameters()` with a uniform `lr=1e-3`. For feature extraction this is benign (frozen params don't update). But if any backbone layers are later unfrozen, the same high lr will destroy pretrained features. Correct: `optim.Adam(model.fc.parameters(), lr=1e-3)` for pure feature extraction.

**Error 2:** Same-lr problem stated more explicitly — no differential LR setup. If this code is extended to partial fine-tuning, catastrophic forgetting is guaranteed.

**Error 3:** `model.train()` is never called before the training loop. The comment says the model was set to evaluation mode earlier. BatchNorm uses running statistics (wrong for training) and Dropout is inactive. `model.train()` must be called at the start of each epoch.

</details>

## Summary

> **In one sentence each:**

1. Why does a small, different-domain dataset call for feature extraction rather than full fine-tuning?
2. What is the correct learning rate ratio between backbone and head when partial fine-tuning?
3. Which two bugs in the pre-written AI code would produce wrong training behaviour without raising an error?


<details>
<summary>🔑 Reveal summary answers</summary>

1. **Small, different-domain dataset → feature extraction:** With only 1,200 images from a domain ImageNet has never seen (medical X-rays), fine-tuning the full backbone risks overwriting pretrained low-level features with noise; freezing the backbone and training only the head preserves the generic feature extractor while adapting the classifier.

2. **Correct lr ratio — backbone vs head:** The backbone should use a learning rate at least 10× lower than the head (e.g. 1e-5 vs 1e-3); the backbone's pretrained weights are already near a good optimum and need only minor adjustment, while the newly initialised head must learn from scratch.

3. **Two bugs from the pre-written code:** Single uniform learning rate (no differential lr structure) and missing `model.train()` call before the training loop — both produce wrong training behaviour without raising an error.

</details>